In [24]:
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

In [25]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'
urls = whoAmI()

In [26]:
DATASET_SAVED = "../../data/raw_official_v2"
URL_INDICATORS = "../../data/urls"

In [27]:
# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(DATASET_SAVED, exist_ok=True)
os.makedirs(URL_INDICATORS, exist_ok=True)

In [28]:
# Lấy các bộ dữ liệu đã có
existed = set()
for file in os.listdir(DATASET_SAVED):
    indicator = file.split('.')[0]
    existed.add(indicator)

In [29]:
dataset_indicators = set()
for file in os.listdir(URL_INDICATORS):
    with open(URL_INDICATORS + '/' + file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line in existed:
                continue
            dataset_indicators.add(line)
print("Các indicators:", dataset_indicators)

Các indicators: {'', 'UHC_SERVICE_COVERAGE', 'HCF_SERVICE_READINESS', 'HCF_BACKUP_POWER', 'HRH_DENSITY_10K', 'HWF_0001', 'HWF_0002', 'UHC_SCI', 'UHC_INDEX_REVISED', 'HCF_INTERNET_ACCESS', 'WHS6_15', 'HWF_0006', 'HCF_MEDICAL_EQUIPMENT', 'HCF_WATER_SUPPLY', 'HCF_BASIC_AMENITIES'}


In [30]:
def request_data(indicator):
    resp = requests.get(urls.getIndData + '/' + indicator)
    data = json.loads(resp.content)["value"]
    fields = ['ParentLocationCode', 'SpatialDim', 'Value', 'NumericValue', 'TimeDimensionBegin', 'TimeDimensionEnd', 'TimeDimensionValue', 'TimeDimType', 'TimeDim', 'IndicatorCode', 'Date']

    records = []
    for row in data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)
    
    with open(DATASET_SAVED + '/' + indicator + '.json', "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [31]:
# Thử chương trình lấy một bộ dữ liệu
# request_data("SA_0000001403")

In [32]:
# Hàm lấy hết dữ liệu
def retrieve_all():
    for indicator in tqdm(dataset_indicators):
        try:
            request_data(indicator)
        except Exception as e:
            print("Error:", e)

In [33]:
#retrieve_all()

In [34]:
retrieve_all()

  7%|▋         | 1/15 [00:03<00:43,  3.12s/it]

Error: 'ParentLocationCode'


 13%|█▎        | 2/15 [00:04<00:24,  1.90s/it]

Error: Expecting value: line 1 column 1 (char 0)


 20%|██        | 3/15 [00:05<00:17,  1.46s/it]

...

Error: Expecting value: line 1 column 1 (char 0)


 93%|█████████▎| 14/15 [00:25<00:01,  1.58s/it]

Error: Expecting value: line 1 column 1 (char 0)


100%|██████████| 15/15 [00:26<00:00,  1.76s/it]

Error: Expecting value: line 1 column 1 (char 0)


In [35]:
# Biến chỉ định lưu trữ
CONCAT_GROUP = "../../data/raw_concat"
os.makedirs(CONCAT_GROUP, exist_ok=True)

In [36]:
# Tiến hành tổ chức và nhóm lại dữ liệu vào file gốc theo các nhãn
def prepare_data_through_group(group):
    # Group là nhãn dữ liệu, tên thư mục
    target = group.split('.')[0]
    objectives = set()

    with open(URL_INDICATORS + '/' + group, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            objectives.add(line)
    
    array_data = []
    for file in tqdm(os.listdir(DATASET_SAVED)):
        _label = file.split('.')[0]
        if _label in objectives:
            # Đúng nhãn dữ liệu, tiến hành load ra và ghép lại thành một file hoàn chỉnh
            with open(DATASET_SAVED + '/' + file, "rb") as f:
                data = json.load(f)
                
                # Tiến hành nối dữ liệu vào file
            array_data = array_data + data
                
    concated_result = CONCAT_GROUP + '/' + target + '.json'
    
    print("Tổng số điểm dữ liệu là:", len(array_data))
    with open(concated_result, "w", encoding="utf-8") as f:
        json.dump(array_data, f, ensure_ascii=False, indent=2)
    print("Hoàn thành xử lý:", target)

In [37]:
# Chạy xử lí chỉ số BMI
prepare_data_through_group("BMI.txt")

100%|██████████| 555/555 [00:01<00:00, 389.65it/s]


Tổng số điểm dữ liệu là: 453077
Hoàn thành xử lý: BMI


In [38]:
# CHạy xử lí chỉ số Diabetes
prepare_data_through_group("diabetes.txt")

100%|██████████| 555/555 [00:00<00:00, 804.27it/s]


Tổng số điểm dữ liệu là: 211564
Hoàn thành xử lý: diabetes


In [39]:
# Chạy xử lí tiêu thụ cồn
prepare_data_through_group("alcohol_consumption.txt")

100%|██████████| 555/555 [00:01<00:00, 288.09it/s] 


Tổng số điểm dữ liệu là: 301920
Hoàn thành xử lý: alcohol_consumption


In [40]:
# Chạy xử lí air_pollution
prepare_data_through_group("air_pollution.txt")

100%|██████████| 555/555 [00:02<00:00, 246.90it/s]


Tổng số điểm dữ liệu là: 491076
Hoàn thành xử lý: air_pollution


In [41]:
# Chạy xử lí cholesterol
prepare_data_through_group("cholesterol.txt")

100%|██████████| 555/555 [00:00<00:00, 1758.63it/s]


Tổng số điểm dữ liệu là: 141804
Hoàn thành xử lý: cholesterol


In [42]:
# Chạy xử lí Glucose
prepare_data_through_group("glucose.txt")

100%|██████████| 555/555 [00:00<00:00, 12737.27it/s]

Tổng số điểm dữ liệu là: 21630


Hoàn thành xử lý: glucose


In [43]:
# Chạy xử lí hoạt động thể chất
prepare_data_through_group("physical_activities.txt")

100%|██████████| 555/555 [00:00<00:00, 6195.26it/s]

Tổng số điểm dữ liệu là: 35523


Hoàn thành xử lý: physical_activities


In [44]:
# Chạy xử lí tiếp cận cơ sở y tế
prepare_data_through_group("infrastructure.txt")

100%|██████████| 555/555 [00:00<00:00, 6309.22it/s]

Tổng số điểm dữ liệu là: 20546


Hoàn thành xử lý: infrastructure


In [45]:
# Chạy xử lí thuốc lá
prepare_data_through_group("tobacco.txt")

100%|██████████| 555/555 [00:06<00:00, 80.08it/s] 


Tổng số điểm dữ liệu là: 704483
Hoàn thành xử lý: tobacco


In [46]:
# Chạy xử lí bệnh tim mạch
prepare_data_through_group("cardiovascular_diseases.txt")

100%|██████████| 555/555 [00:00<00:00, 1039.36it/s]


Tổng số điểm dữ liệu là: 236876
Hoàn thành xử lý: cardiovascular_diseases
